# Optional (phase 2): Real terrain via BlenderGIS

**Status: first draft, built from BlenderGIS's actual source code, not yet
run end-to-end.** Everything below uses real BlenderGIS APIs (verified by
reading `geoscene.py` and `operators/io_get_dem.py` in the
[BlenderGIS repo](https://github.com/domlysz/BlenderGIS)), not guessed ones,
though nobody has run it in a live Blender session yet. Treat the first run
as a test, not a trusted step, and expect to debug the offset-verification
step in §0 in particular.

This replaces `1a`'s flat, bounding-box-derived `Ground` plane (see
`gml_to_sionna_via_blender.ipynb` §1.2) with real topography, geolocated
using the source CityGML file's own coordinate reference system. This is
"solution 2" from the ground-plane roadmap in that notebook.

## 0. Before you start

You need, inside Blender (in addition to everything `1a` already needs):

1. The **[BlenderGIS add-on](https://github.com/domlysz/BlenderGIS)**
   installed and enabled. Requires Blender 2.83+; see the root README's
   Setup table for which Blender version this team is standardizing on.
2. A free **[OpenTopography](https://opentopography.org) API key**.
   The default DEM query operator uses OpenTopography as its elevation data
   source and requires registering for a key, set once in
   `Edit > Preferences > Add-ons > BlenderGIS > DEM server / Api Key`.
   This is a manual, one-time, per-machine setup step outside this notebook.
3. `1a` run through at least §1.2 (this notebook selects the `Ground`
   object §1.2 creates, to use its bounding box as the terrain query
   extent). Run this *after* §1.2 and *before* the export cells in §1.6/1.7,
   so the terrain ends up in the final exported scene.

**The one thing in here that's a real, unverified assumption and not just
an API-usage question:** §2 below georeferences the Blender scene by telling
BlenderGIS "Blender world (0, 0) = this real-world coordinate, in this CRS."
That's only true if the CityGML importer preserved absolute coordinates on
import. Many CityGML/BIM importers instead *recentre* geometry near the
origin for floating-point precision, applying an offset that isn't
necessarily logged anywhere obvious. Check this before trusting anything
downstream of §2:

- Look at the `min_x`/`min_y` values `1a` §1.2 printed when it built the
  `Ground` plane. If they're already large (hundreds of thousands, roughly
  matching a UTM easting/northing for Hamburg, e.g. easting in the
  565,000 range, northing in the 5,934,000 range for UTM zone 32N/ETRS89),
  the importer kept absolute coordinates and no offset correction is
  needed. If they're small (tens or low hundreds), the importer recentred,
  and you need to find that offset. Check the CityGML add-on's import
  log/console output, or its own settings panel, for anything describing a
  translation/recentring it applied.
- **German municipal CityGML (this Hamburg Stadtmodell data included) often
  declares a *compound* CRS**, e.g. an URN like
  `urn:adv:crs:ETRS89_UTM32*DE_DHHN92_NH` (horizontal datum × vertical
  datum together), not a plain `EPSG:25832`. BlenderGIS expects a plain
  EPSG/proj4 string. §1 below only extracts the horizontal EPSG component
  (`EPSG:25832` for ETRS89/UTM32N, the usual case for Hamburg) and ignores
  the vertical datum. Ground/terrain Z still comes from Blender's own
  coordinates, not from the CRS's height reference. If §1's parsing doesn't
  produce a clean `EPSG:xxxx`, set `EPSG` by hand in §2.

## 1. Read the CRS and envelope origin from the source CityGML file

This reads straight from the original `.gml`/`.citygml` file, the same one
imported into Blender in `1a` §0, not from anything Blender/the importer
exposes, since that's not confirmed to survive import. Plain Python
(`xml.etree`), no `bpy` needed; run it in Blender's Python console/Scripting
tab like everything else in this notebook, for one consistent workflow.

In [ ]:
import xml.etree.ElementTree as ET

CITYGML_SOURCE_PATH = r"C:\path\to\your\source.gml"  # <-- edit: the original CityGML file

ns = {"gml": "http://www.opengis.net/gml"}
# CityGML 2.0 uses GML 3.1.1, whose namespace URI is "http://www.opengis.net/gml"
# (no version suffix). If this file is CityGML 3.0 (GML 3.2), the namespace is
# instead "http://www.opengis.net/gml/3.2". If the search below raises
# "no Envelope found", try that namespace URI instead.

tree = ET.parse(CITYGML_SOURCE_PATH)
root = tree.getroot()

# .find(".//...") returns the FIRST match in document order. This assumes the
# file has one global <gml:boundedBy><gml:Envelope> near the top describing the
# whole dataset's extent (standard for CityGML exports from a WFS/city portal).
# If your file instead only has per-building envelopes nested deeper, this will
# silently grab one building's tiny envelope instead of the dataset's, so
# sanity-check the printed numbers below against what you actually expect.
envelope = root.find(".//gml:boundedBy/gml:Envelope", ns)
if envelope is None:
    raise RuntimeError(
        "No <gml:Envelope> found under <gml:boundedBy>. Either the namespace is "
        "wrong (see the GML 3.2 note above) or this file structures it "
        "differently. Open it in a text editor and look for srsName/"
        "lowerCorner near the top, then set EPSG/LOWER_X/LOWER_Y by hand in "
        "Section 2 instead of relying on this cell."
    )

srs_name = envelope.get("srsName")  # e.g. "EPSG:25832" or a compound URN, see 0.
lower_corner = envelope.find("gml:lowerCorner", ns).text.split()
X, Y = float(lower_corner[0]), float(lower_corner[1])

# GML axis order is a known trap: depending on how strictly a service follows
# the CRS's official axis order, coordinates can come out as (Easting, Northing)
# or (Northing, Easting), and the file doesn't flag which. For UTM (EPSG:25832,
# what Hamburg uses), eastings are always under ~1,000,000 by construction
# (false easting = 500,000), while northings this far north are roughly
# 5,900,000-5,950,000, so a first value over 1,000,000 means the order is
# actually (Northing, Easting) and needs swapping.
if X > 1_000_000 and Y < 1_000_000:
    print("First coordinate > 1,000,000, looks like (Northing, Easting) order, swapping.")
    X, Y = Y, X
LOWER_X, LOWER_Y = X, Y

if "EPSG" in srs_name:
    epsg_code = srs_name.split("EPSG")[-1].strip(":").split("::")[-1].strip(":")
    EPSG = f"EPSG:{epsg_code}"
else:
    EPSG = None  # compound/non-EPSG URN, set this by hand in Section 2, see 0.

print(f"srsName as written in the file: {srs_name}")
print(f"Parsed EPSG (None means: set it by hand, see the note in Section 0): {EPSG}")
print(f"Envelope lower corner (Easting, Northing): X={LOWER_X}, Y={LOWER_Y}")
print("Sanity check: for Hamburg/UTM32N, expect X (easting) roughly 550,000-580,000 "
      "and Y (northing) roughly 5,920,000-5,950,000. If these numbers look wildly "
      "different, something above is wrong. Don't trust this automatically.")

## 2. Georeference the Blender scene

Sets the same custom scene properties BlenderGIS's own `GeoScene` class
manages (`scene["SRID"]`, `scene["crs x"]`, `scene["crs y"]`) directly. That's
what the class does internally, and setting them directly avoids depending
on how the add-on happens to be importable as a Python module.

**Do not skip the offset check in §0 before running this.**

In [ ]:
import bpy

# From Section 1: override here if you parsed a different file, or if
# Section 1 couldn't determine a clean EPSG code (see its printed output).
EPSG = "EPSG:25832"       # <-- edit if needed
LOWER_X = 565000.0        # <-- edit if needed
LOWER_Y = 5934000.0       # <-- edit if needed

# See the offset-verification note in "0. Before you start". Do not leave
# this at (0, 0) without checking first.
BLENDER_ORIGIN_OFFSET = (0.0, 0.0)  # <-- verify, then edit if needed

scn = bpy.context.scene
scn["SRID"] = EPSG
scn["crs x"] = LOWER_X + BLENDER_ORIGIN_OFFSET[0]
scn["crs y"] = LOWER_Y + BLENDER_ORIGIN_OFFSET[1]

print(f"Scene georeferenced: SRID={scn['SRID']}, origin=({scn['crs x']}, {scn['crs y']})")
print("Sanity check: open the 3D viewport sidebar (press N) and look for a "
      "'Geoscene' / 'GIS' tab. BlenderGIS should show this same CRS and "
      "origin there if it picked up the georeferencing correctly.")

## 3. Fetch and import terrain (DEM)

Selects the `Ground` object from `1a` §1.2 and uses its bounding box as the
query extent, then calls BlenderGIS's DEM-fetch operator directly
(`'EXEC_DEFAULT'` skips its interactive dialog, since the DEM
server/API key are read from the add-on's saved preferences either way,
see §0).

In [ ]:
import bpy

ground = bpy.data.objects.get("Ground")
if ground is None:
    raise RuntimeError("No 'Ground' object found. Run 1a Section 1.2 first.")

for obj in bpy.context.selected_objects:
    obj.select_set(False)
ground.select_set(True)
bpy.context.view_layer.objects.active = ground

result = bpy.ops.importgis.dem_query('EXEC_DEFAULT')
print("DEM import result:", result)
print("If this returned {'CANCELLED'}, check the Info/System Console for the "
      "actual error. Most likely causes: scene not georeferenced (Section 2 "
      "didn't run or BlenderGIS didn't pick it up), missing OpenTopography API "
      "key (Section 0), or the query extent falling outside the DEM source's "
      "coverage.")

## 4. Sanity-check before trusting this

Nothing here confirms the terrain actually lines up with the buildings.
That needs eyes on it. At minimum:
- Render or preview the scene (same pattern as `1a` §3/Section 3) and look
  for buildings floating above or sinking into the new terrain mesh.
- If the offset in §0/§2 was wrong, this is where it'll show up: buildings
  correctly shaped but sitting in the wrong place relative to the terrain,
  or a terrain patch that's clearly the wrong stretch of Hamburg.

If this is wrong, it's very likely the `BLENDER_ORIGIN_OFFSET` assumption
in §0/§2, not the DEM fetch itself. Go back and actually determine the
CityGML importer's recentring behavior rather than guessing at offset
values.